In [ ]:
import pandas as pd
import requests
import time
from sklearn.linear_model import LinearRegression

CHANNEL_ID = "3304823"
WRITE_API_KEY = "VVHKBIU17Z53C9J9"

# Load data
url = f"https://api.thingspeak.com/channels/{CHANNEL_ID}/feeds.csv"
df = pd.read_csv(url)

# Clean
df = df[['created_at','field3','field4','field5','field6']]
df.columns = ['time','co2','co','dust','aqi']
df = df.dropna()

df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(by='time').tail(300)

# Future AQI (10 min)
df['aqi_future'] = df['aqi'].shift(-5)
df = df.dropna()

# Train
X = df[['co2','co','dust']]
y = df['aqi_future']

model = LinearRegression()
model.fit(X,y)

# Predict
# Predict
pred = model.predict([[latest['co2'], latest['co'], latest['dust']]])[0]
pred = round(float(pred), 2)

print("Predicted AQI:", pred)

# WAIT (important)
time.sleep(60)

# Upload
write_url = f"https://api.thingspeak.com/update?api_key={WRITE_API_KEY}&field7={pred}"
res = requests.get(write_url)

print("Uploaed response:", res.text)